# Third-Party Billing Reconciliation: Comprehensive Audit & Remediation

**Analysis Date:** 2026-08-31  
**Purpose:** Audit 50,078 exception rows ($32.6M estimated impact), validate data quality, detect root-cause patterns, and recommend remediation prioritization.

**Key Baseline Metrics:**
- **Total Exception Rows:** 50,078
- **Estimated Impact:** $32,642,484.83
- **Non-Clear Rows:** 50,078 (outcome_flag != 'Clear')
- **Top Vendor by Impact:** SentinelOne ($12.9M), Webroot ($7.0M), KeepIT ($4.8M)

## Workflow Overview
1. Load and inspect the queue export CSV
2. Clean and normalize data types
3. Validate and rebuild Case IDs
4. Audit exception type vs outcome flag consistency
5. Apply canonical bucket mapping
6. Compute exposure rollups and persistence risk
7. Build actionable queue views
8. Export cleaned audit outputs for remediation

In [1]:
# Section 1: Load Libraries and Configuration

import sys
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime, timedelta
import warnings

# Add projects to path for access to TEMPLATES module
sys.path.insert(0, r'c:/Users/Nate.Fold/projects')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 120)

# File paths
WORKSPACE_ROOT = Path(r'c:\Users\Nate.Fold\projects')
ATTACHMENT_CSV = WORKSPACE_ROOT / 'Downloads' / '2026-09-01T03-05_export.csv'
OUTPUT_DIR = WORKSPACE_ROOT / 'reconciliation_audit_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ Libraries loaded")
print(f"✓ Attachment CSV: {ATTACHMENT_CSV}")
print(f"✓ Output directory: {OUTPUT_DIR}")

✓ Libraries loaded
✓ Attachment CSV: c:\Users\Nate.Fold\projects\Downloads\2026-09-01T03-05_export.csv
✓ Output directory: c:\Users\Nate.Fold\projects\reconciliation_audit_outputs


In [14]:
# Section 2: Read Attachment CSV and Snowflake Full Dataset

# Load attached export (top-250 exceptions from queue)
if ATTACHMENT_CSV.exists():
    df_attachment = pd.read_csv(ATTACHMENT_CSV)
    print(f"✓ Loaded attachment CSV: {len(df_attachment)} rows, {len(df_attachment.columns)} columns")
    print(f"\nColumns: {list(df_attachment.columns)}")
    print(f"\nData types:\n{df_attachment.dtypes}")
else:
    print(f"⚠ Attachment not found: {ATTACHMENT_CSV}")
    df_attachment = None

# Also load FULL dataset from Snowflake for comprehensive analysis
from TEMPLATES.Python.connection import get_snowflake_connection, fetch_dataframe

conn = get_snowflake_connection(role='DEVELOPER', warehouse='REPORTING_WH', database='ANALYTICS_DEV', schema='DBT_NFOLD_TRANSFORMATION')

print("\n\n--- Loading FULL exception dataset from Snowflake ---")
q = """
SELECT 
  vendor,
  billing_month,
  inv_id,
  sf_id,
  billing_type,
  vendor_partner_name,
  vendor_product,
  sku_match_group,
  cw_skus,
  zuora_skus,
  marketplace_skus,
  billing_source_mix,
  api_quantity,
  avg_api_quantity,
  vendor_quantity,
  vendor_unit_price,
  vendor_amount,
  zuora_quantity,
  zuora_unit_price,
  zuora_amount,
  marketplace_quantity,
  marketplace_unit_price,
  marketplace_amount,
  total_billing_quantity,
  total_billing_amount,
  qty_delta,
  abs_qty_delta,
  amount_delta,
  abs_amount_delta,
  cw_margin_pct,
  has_discount,
  duplicate_billing_flag,
  outcome_flag,
  investigation_reason,
  exception_type,
  est_dollar_impact,
  vendor_source_row_count,
  api_amount,
  avg_api_amount,
  product_display,
  partner_display_name,
  action_needed,
  is_leakage,
  is_finance_queue,
  is_ops_queue,
  is_timing_queue,
  is_clear,
  case_id
FROM THIRD_PARTY_RECON_OUTPUT_PROD
WHERE outcome_flag != 'Clear' OR outcome_flag IS NULL
ORDER BY est_dollar_impact DESC NULLS LAST
"""

df = fetch_dataframe(q, conn=conn)
print(f"✓ Loaded full dataset from Snowflake: {len(df)} rows")

conn.close()

# Quick preview
print(f"\nDataset shape: {df.shape}")
print(f"\nNull counts (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))
print(f"\nSample rows (top 5):")
df.head(5)

⚠ Attachment not found: c:\Users\Nate.Fold\projects\Downloads\2026-09-01T03-05_export.csv


--- Loading FULL exception dataset from Snowflake ---
✓ Loaded full dataset from Snowflake: 50078 rows

Dataset shape: (50078, 48)

Null counts (top 10):
BILLING_TYPE              50078
API_QUANTITY              38247
AVG_API_QUANTITY          36929
MARKETPLACE_UNIT_PRICE    35903
MARKETPLACE_SKUS          35626
ZUORA_UNIT_PRICE          26904
ZUORA_SKUS                26763
CW_SKUS                   26029
INV_ID                    24589
CW_MARGIN_PCT             23996
dtype: int64

Sample rows (top 5):


,VENDOR,BILLING_MONTH,INV_ID,SF_ID,BILLING_TYPE,VENDOR_PARTNER_NAME,VENDOR_PRODUCT,SKU_MATCH_GROUP,CW_SKUS,ZUORA_SKUS,MARKETPLACE_SKUS,BILLING_SOURCE_MIX,API_QUANTITY,AVG_API_QUANTITY,VENDOR_QUANTITY,VENDOR_UNIT_PRICE,VENDOR_AMOUNT,ZUORA_QUANTITY,ZUORA_UNIT_PRICE,ZUORA_AMOUNT,MARKETPLACE_QUANTITY,MARKETPLACE_UNIT_PRICE,MARKETPLACE_AMOUNT,TOTAL_BILLING_QUANTITY,TOTAL_BILLING_AMOUNT,QTY_DELTA,ABS_QTY_DELTA,AMOUNT_DELTA,ABS_AMOUNT_DELTA,CW_MARGIN_PCT,HAS_DISCOUNT,DUPLICATE_BILLING_FLAG,OUTCOME_FLAG,INVESTIGATION_REASON,EXCEPTION_TYPE,EST_DOLLAR_IMPACT,VENDOR_SOURCE_ROW_COUNT,API_AMOUNT,AVG_API_AMOUNT,PRODUCT_DISPLAY,PARTNER_DISPLAY_NAME,ACTION_NEEDED,IS_LEAKAGE,IS_FINANCE_QUEUE,IS_OPS_QUEUE,IS_TIMING_QUEUE,IS_CLEAR,CASE_ID
0,SentinelOne,2026-01-01,MULTI_INV_54,ACT-00059853,None,NuMSP,S1ES-CTL-EN-T2-SA | S1ES-CTL-EN-T9-SA,CONTROL,"CUSERVOTHER300440PAR,CUSERVOTHER300450PAR,CUSERVOTHR300520EPSB,CWSENTINEL1-CONTROL,M2MSEROTHR300520EPSB,SENTINELONE-...","CUSERVOTHR300520EPSB,M2MSEROTHR300520EPSB,SENTINELONE-CONTROL",None,ZUORA_ONLY,36818.0,37482.548387,0.0,0.720000,0.00,36494.0,4.228125,128662.75000,0.0,NaN,0.0,36494.0,128662.75000,36494.0,36494.0,128662.75000,128662.75000,100.0,FALSE,FALSE,"CW Billing, No Vendor Billing",CW bills material qty for this partner but SentinelOne vendor file reports ~0. Suggests SentinelOne-side site attrib...,Clear,128662.75000,1,26508.96,26987.434839,Control,Secur-Serv,None,False,False,False,False,True,SentinelOne|ACT-00059853|CONTROL|2026-01|Clear
1,KeepIT,2026-05-01,None,None,None,ConnectWise (Continuum) - Consolidated,KI-M365-FUL,None,"BB-3Y-PROMO-BUNDLE,BC-3Y-PROMO-OFC365UR,CMS-3P-UMM-BCDR-SAAS-OFC365UR,CMS-EG-BDR-SOLP-SAAS-RMSBM365,CULCSAS200310OFF...",BB-3Y-PROMO-BUNDLE,None,ZUORA_ONLY,NaN,NaN,366961.0,0.758571,278365.90,278760.0,1.434578,399902.99692,0.0,NaN,0.0,278760.0,399902.99692,-88201.0,88201.0,121537.09692,121537.09692,30.4,FALSE,FALSE,Unmapped Partner,Vendor usage has KeepIT partner GUID but no resolved ACT account from governed CMS crosswalk.,Clear,121537.09692,1,0.00,0.000000,KI-M365-FUL,ConnectWise (Continuum) - Consolidated,None,False,False,False,False,True,KeepIT||KI-M365-FUL|2026-05|Clear
2,KeepIT,2026-02-01,None,None,None,ConnectWise (Continuum) - Consolidated,KI-M365-FUL,None,"BB-3Y-PROMO-BUNDLE,BC-3Y-PROMO-OFC365UR,CMS-3P-UMM-BCDR-SAAS-OFC365UR,CMS-EG-BDR-SOLP-SAAS-RMSBM365,CULCSAS200310OFF...",BB-3Y-PROMO-BUNDLE,None,ZUORA_ONLY,NaN,NaN,365214.0,0.697382,254693.70,262614.0,1.417598,372281.14764,0.0,NaN,0.0,262614.0,372281.14764,-102600.0,102600.0,117587.44764,117587.44764,31.6,FALSE,FALSE,Unmapped Partner,Vendor usage has KeepIT partner GUID but no resolved ACT account from governed CMS crosswalk.,Clear,117587.44764,1,0.00,0.000000,KI-M365-FUL,ConnectWise (Continuum) - Consolidated,None,False,False,False,False,True,KeepIT||KI-M365-FUL|2026-02|Clear
3,KeepIT,2026-03-01,None,None,None,ConnectWise (Continuum) - Consolidated,KI-M365-FUL,None,"BB-3Y-PROMO-BUNDLE,BC-3Y-PROMO-OFC365UR,CMS-3P-UMM-BCDR-SAAS-OFC365UR,CMS-EG-BDR-SOLP-SAAS-RMSBM365,CULCSAS200310OFF...",BB-3Y-PROMO-BUNDLE,None,ZUORA_ONLY,NaN,NaN,336652.0,0.774987,260900.95,257786.0,1.439684,371130.30420,0.0,NaN,0.0,257786.0,371130.30420,-78866.0,78866.0,110229.35420,110229.35420,29.7,FALSE,FALSE,Unmapped Partner,Vendor usage has KeepIT partner GUID but no resolved ACT account from governed CMS crosswalk.,Clear,110229.35420,1,0.00,0.000000,KI-M365-FUL,ConnectWise (Continuum) - Consolidated,None,False,False,False,False,True,KeepIT||KI-M365-FUL|2026-03|Clear
4,KeepIT,2026-04-01,None,None,None,ConnectWise (Continuum) - Consolidated,KI-M365-FUL,None,"BB-3Y-PROMO-BUNDLE,BC-3Y-PROMO-OFC365UR,CMS-3P-UMM-BCDR-SAAS-OFC365UR,CMS-EG-BDR-SOLP-SAAS-RMSBM365,CULCSAS200310OFF...",BB-3Y-PROMO-BUNDLE,None,ZUORA_ONLY,NaN,NaN,368513.0,0.756801,278891.10,268300.0,1.436964,385537.48303,0.0,NaN,0.0,268300.0,385537.48303,-100213.0,100213.0,106646.38303,106646.38303,27.7,FALSE,FALSE,Unmapped Partner,Vendor usage has KeepIT partner GUID but no resolve

## Section 3: Data Quality Assessment & Normalization

Clean and normalize all key fields for downstream matching and analysis.

In [15]:
# Data normalization and quality checks

df = df.copy()

# Rename columns to lowercase for consistency
df.columns = df.columns.str.lower()

# 1. Parse billing_month to datetime
df['billing_month'] = pd.to_datetime(df['billing_month'], errors='coerce')

# 2. Normalize numeric fields
df['est_dollar_impact'] = pd.to_numeric(df['est_dollar_impact'], errors='coerce')
df['vendor_amount'] = pd.to_numeric(df['vendor_amount'], errors='coerce')
df['zuora_amount'] = pd.to_numeric(df['zuora_amount'], errors='coerce')
df['marketplace_amount'] = pd.to_numeric(df['marketplace_amount'], errors='coerce')
df['total_billing_amount'] = pd.to_numeric(df['total_billing_amount'], errors='coerce')
df['amount_delta'] = pd.to_numeric(df['amount_delta'], errors='coerce')
df['abs_amount_delta'] = pd.to_numeric(df['abs_amount_delta'], errors='coerce')

# 3. Normalize quantities
df['vendor_quantity'] = pd.to_numeric(df['vendor_quantity'], errors='coerce')
df['zuora_quantity'] = pd.to_numeric(df['zuora_quantity'], errors='coerce')
df['marketplace_quantity'] = pd.to_numeric(df['marketplace_quantity'], errors='coerce')
df['total_billing_quantity'] = pd.to_numeric(df['total_billing_quantity'], errors='coerce')
df['api_quantity'] = pd.to_numeric(df['api_quantity'], errors='coerce')
df['avg_api_quantity'] = pd.to_numeric(df['avg_api_quantity'], errors='coerce')
df['qty_delta'] = pd.to_numeric(df['qty_delta'], errors='coerce')
df['abs_qty_delta'] = pd.to_numeric(df['abs_qty_delta'], errors='coerce')

# 4. String normalization: trim whitespace, uppercase for SKUs, title case for names
for col in ['vendor', 'vendor_product', 'vendor_partner_name', 'partner_display_name', 
            'cw_skus', 'zuora_skus', 'product_display', 'action_needed']:
    if col in df.columns:
        df[col] = df[col].fillna('').str.strip()

# 5. Standardize empty/unmapped values
for col in ['partner_display_name', 'vendor_partner_name', 'action_needed', 'sf_id']:
    if col in df.columns:
        df[col] = df[col].replace(['', '(unmapped)', 'unmapped', 'N/A'], '')

# 6. Create normalized key columns
df['vendor_norm'] = df['vendor'].str.upper().str.replace(r'\s+', '_', regex=True)
df['sf_id_norm'] = df['sf_id'].fillna('').str.upper()
df['partner_norm'] = df['partner_display_name'].fillna('').str.upper().str.strip()
df['month_key'] = df['billing_month'].dt.strftime('%Y-%m')
df['match_key'] = df['vendor_norm'] + '|' + df['sf_id_norm'] + '|' + df['month_key']

print("✓ Data normalized")
print(f"\nData type summary after normalization:")
print(df[['vendor', 'sf_id', 'billing_month', 'est_dollar_impact', 'vendor_quantity', 'outcome_flag']].dtypes)
print(f"\nSample normalized keys:")
print(df[['vendor_norm', 'sf_id_norm', 'month_key', 'match_key']].head(3))

✓ Data normalized

Data type summary after normalization:
vendor                       object
sf_id                        object
billing_month        datetime64[ns]
est_dollar_impact           float64
vendor_quantity             float64
outcome_flag                 object
dtype: object

Sample normalized keys:
   vendor_norm    sf_id_norm month_key                         match_key
0  SENTINELONE  ACT-00059853   2026-01  SENTINELONE|ACT-00059853|2026-01
1       KEEPIT                 2026-05                   KEEPIT||2026-05
2       KEEPIT                 2026-02                   KEEPIT||2026-02


## Section 4: Quality Issues Discovered

In [16]:
# Quality assessment findings

quality_findings = {}

# A. Missing and Conflicting Salesforce IDs
print("=" * 80)
print("A. SALESFORCE ID QUALITY")
print("=" * 80)

missing_sf_id = df[df['sf_id'].isna() | (df['sf_id'] == '')].shape[0]
print(f"\nRows with missing SF_ID: {missing_sf_id:,} ({100*missing_sf_id/len(df):.1f}%)")

sf_id_duplicates = df.groupby('sf_id_norm').size().sort_values(ascending=False)
print(f"Total unique SF_IDs: {len(sf_id_duplicates)}")
print(f"\nTop 10 SF_IDs by row count:")
print(sf_id_duplicates.head(10))

quality_findings['missing_sf_id'] = missing_sf_id
quality_findings['unique_sf_ids'] = len(sf_id_duplicates)

# B. Partner Name Conflicts
print("\n" + "=" * 80)
print("B. PARTNER NAME QUALITY")
print("=" * 80)

missing_partner = df[df['partner_display_name'].isna() | (df['partner_display_name'] == '')].shape[0]
print(f"\nRows with missing partner name: {missing_partner:,} ({100*missing_partner/len(df):.1f}%)")

# Check if same SF_ID has multiple partner names
sf_partner_conflicts = df.groupby('sf_id_norm')['partner_display_name'].nunique()
conflicts_count = (sf_partner_conflicts > 1).sum()
print(f"\nSF_IDs with conflicting partner names: {conflicts_count:,}")

quality_findings['missing_partner_names'] = missing_partner
quality_findings['sf_id_partner_conflicts'] = conflicts_count

# C. SKU Quality
print("\n" + "=" * 80)
print("C. SKU MAPPING QUALITY")
print("=" * 80)

missing_cw_sku = df[df['cw_skus'].isna() | (df['cw_skus'] == '')].shape[0]
missing_vendor_sku = df[df['product_display'].isna() | (df['product_display'] == '')].shape[0]
both_missing = df[((df['cw_skus'].isna() | (df['cw_skus'] == '')) & 
                   (df['product_display'].isna() | (df['product_display'] == '')))].shape[0]

print(f"\nRows with missing CW SKU: {missing_cw_sku:,}")
print(f"Rows with missing Vendor SKU: {missing_vendor_sku:,}")
print(f"Rows missing BOTH: {both_missing:,}")

# D. Amount and Quantity Mismatches
print("\n" + "=" * 80)
print("D. AMOUNT & QUANTITY MISMATCHES")
print("=" * 80)

qty_delta_high = (df['abs_qty_delta'] > 10000).sum()
amt_delta_high = (df['abs_amount_delta'] > 5000).sum()

print(f"\nRows with high quantity delta (|qty_delta| > 10k): {qty_delta_high:,}")
print(f"Rows with high amount delta (|amt_delta| > $5k): {amt_delta_high:,}")

print(f"\nAmount delta statistics:")
print(df[['amount_delta', 'abs_amount_delta']].describe())

quality_findings['high_qty_deltas'] = qty_delta_high
quality_findings['high_amt_deltas'] = amt_delta_high

# E. Outcome Flag vs Exception Type Consistency
print("\n" + "=" * 80)
print("E. OUTCOME FLAG vs EXCEPTION TYPE CONSISTENCY")
print("=" * 80)

inconsistent_rows = []
for idx, row in df.iterrows():
    flag = row['outcome_flag']
    exc_type = row['exception_type']
    
    # Check if they disagree on severity
    if flag == 'Clear' and exc_type not in ['Clear', 'Marketplace Billing Delay', 'Known Discount / Bundle']:
        inconsistent_rows.append(idx)
    elif flag in ['Unmapped Partner', 'Vendor SKU, No CW SKU', 'CW SKU, No Vendor SKU'] and exc_type == 'Clear':
        inconsistent_rows.append(idx)

print(f"\nRows with outcome/exception inconsistency: {len(inconsistent_rows):,}")
print(f"% of total: {100*len(inconsistent_rows)/len(df):.1f}%")

if inconsistent_rows:
    sample_incon = df.loc[inconsistent_rows[:5], ['vendor', 'sf_id', 'outcome_flag', 'exception_type', 'est_dollar_impact']]
    print(f"\nSample inconsistent rows:")
    print(sample_incon.to_string())

quality_findings['outcome_inconsistencies'] = len(inconsistent_rows)

print(f"\n✓ Quality assessment complete")
print(f"\nSummary of findings: {quality_findings}")

A. SALESFORCE ID QUALITY

Rows with missing SF_ID: 110 (0.2%)
Total unique SF_IDs: 3890

Top 10 SF_IDs by row count:
sf_id_norm
ACT-00059853    858
ACT-00088629    740
ACT-00174754    697
ACT-00206346    485
ACT-00252312    271
ACT-00228270    248
ACT-00290884    247
ACT-00060280    150
ACT-00118437    150
ACT-00020131    148
dtype: int64

B. PARTNER NAME QUALITY

Rows with missing partner name: 237 (0.5%)

SF_IDs with conflicting partner names: 5

C. SKU MAPPING QUALITY

Rows with missing CW SKU: 26,068
Rows with missing Vendor SKU: 0
Rows missing BOTH: 0

D. AMOUNT & QUANTITY MISMATCHES

Rows with high quantity delta (|qty_delta| > 10k): 1,078
Rows with high amount delta (|amt_delta| > $5k): 1,054

Amount delta statistics:
        amount_delta  abs_amount_delta
count   50078.000000      50078.000000
mean      360.850164        651.832837
std      3189.954682       3143.426758
min    -23249.000000          0.000000
25%       -39.200000         14.100000
50%         1.020000         87

## Section 5: Root Cause Classification

Assign each exception group a primary root cause using the precedence hierarchy.

In [17]:
# Root cause classification logic

def classify_root_cause(row):
    """
    Precedence hierarchy to assign ONE primary root cause per row.
    Upstream failures suppress downstream symptoms.
    """
    
    flag = row.get('outcome_flag', '')
    exc_type = row.get('exception_type', '')
    has_sf_id = row.get('sf_id') and str(row['sf_id']).strip()
    has_partner = row.get('partner_display_name') and str(row['partner_display_name']).strip()
    has_cw_sku = row.get('cw_skus') and str(row['cw_skus']).strip()
    has_vendor_sku = row.get('product_display') and str(row['product_display']).strip()
    
    # 1. PARTNER-MAP FAILURES (highest precedence)
    if exc_type == 'Unmapped Partner' or not has_sf_id:
        return 'Partner-map failure'
    
    # 2. SKU-MAP FAILURES
    if exc_type == 'Vendor SKU, No CW SKU':
        return 'SKU-map failure (vendor SKU missing)'
    if exc_type == 'CW SKU, No Vendor SKU':
        return 'SKU-map failure (CW SKU missing)'
    if not has_vendor_sku and row.get('vendor_amount', 0) > 0:
        return 'SKU-map failure (vendor SKU missing)'
    
    # 3. DISABLED or EXPIRED MAPPINGS
    if exc_type == 'Disabled Partner SKU':
        return 'Disabled/expired mapping'
    
    # 4. QUANTITY MISMATCHES (emphasize for specific vendors like KeepIT)
    qty_delta = abs(row.get('qty_delta', 0) or 0)
    if qty_delta > 100:  # Significant quantity variance
        return 'Quantity mismatch'
    
    # 5. PRICE / UNIT-OF-MEASURE MISMATCHES
    amt_delta = abs(row.get('amount_delta', 0) or 0)
    if amt_delta > 1000 and qty_delta < 10:
        return 'Rate or price mismatch'
    
    # 6. PRORATION OR TIMING DIFFERENCES
    if exc_type == 'Marketplace Billing Delay':
        return 'Proration/timing difference'
    
    # 7. DUPLICATE SOURCE RECORDS
    if row.get('duplicate_billing_flag'):
        return 'Duplicate source record'
    
    # 8. API / VENDOR USAGE CONFIRMATION
    if exc_type == 'API Usage, Insufficient CW Billing':
        return 'Genuine missing CW billing (API-confirmed)'
    
    # 9. GENUINE MISSING BILLING
    if exc_type == 'Vendor Billing, No CW Billing':
        return 'Genuine missing CW billing (vendor-confirmed)'
    if exc_type == 'CW Billing, No Vendor Billing':
        return 'Genuine missing vendor billing (or stale)'
    
    # 10. INSUFFICIENT BILLING (CW below vendor by >25%)
    if exc_type == 'Vendor Billing, Insufficient CW Billing':
        return 'Insufficient CW billing (margin gap)'
    
    return 'Other unresolved issue'

df['root_cause'] = df.apply(classify_root_cause, axis=1)

# Summarize root causes
print("=" * 80)
print("ROOT CAUSE DISTRIBUTION")
print("=" * 80)

root_cause_summary = df.groupby('root_cause').agg({
    'est_dollar_impact': ['count', 'sum', 'mean'],
    'vendor': lambda x: x.value_counts().index[0] if len(x) > 0 else 'N/A'
}).round(2)

root_cause_summary.columns = ['Row Count', 'Total Impact', 'Avg Impact', 'Top Vendor']
root_cause_summary = root_cause_summary.sort_values('Total Impact', ascending=False)

print("\n" + root_cause_summary.to_string())

print(f"\n✓ Root cause classification complete")
print(f"\nTotal unique root causes: {df['root_cause'].nunique()}")

ROOT CAUSE DISTRIBUTION

                                      Row Count  Total Impact  Avg Impact   Top Vendor
root_cause                                                                            
Quantity mismatch                         19866   28419026.21     1430.54      Webroot
Duplicate source record                   26629    2252466.07       84.59      Webroot
Rate or price mismatch                      331     683822.19     2065.93   Proofpoint
Partner-map failure                         110     660355.11     6003.23      Webroot
SKU-map failure (CW SKU missing)            883     332535.09      376.60       KeepIT
SKU-map failure (vendor SKU missing)       1793     209499.17      116.84  SentinelOne
Disabled/expired mapping                    410      80391.10      196.08      Acronis
Proration/timing difference                  56       4389.87       78.39      Acronis

✓ Root cause classification complete

Total unique root causes: 8


## Section 6: Persistence Analysis

Identify chronic issues recurring across multiple billing months.

In [18]:
# Persistence analysis

print("=" * 80)
print("PERSISTENCE ANALYSIS - Chronic Cases Recurring Across Months")
print("=" * 80)

# Group by stable account-product-root-cause key
persistence_key = df.groupby(['vendor', 'sf_id', 'product_display', 'root_cause']).agg({
    'billing_month': ['count', 'min', 'max'],
    'est_dollar_impact': 'sum',
    'abs_amount_delta': 'mean',
    'case_id': lambda x: x.iloc[0]  # Pick first case_id as representative
}).round(2)

persistence_key.columns = ['Month_Count', 'First_Month', 'Last_Month', 'Total_Impact', 'Avg_Amount_Delta', 'Sample_Case_ID']
persistence_key = persistence_key.sort_values('Total_Impact', ascending=False)

# Identify truly chronic cases (6+ months)
chronic_cases = persistence_key[persistence_key['Month_Count'] >= 6]

print(f"\nChronic cases (6+ months): {len(chronic_cases)}")
print(f"Total rows in chronic cases: {chronic_cases['Month_Count'].sum():,}")
print(f"Total impact from chronic cases: ${chronic_cases['Total_Impact'].sum():,.2f}")
print(f"% of total exception impact: {100*chronic_cases['Total_Impact'].sum()/df['est_dollar_impact'].sum():.1f}%")

print(f"\n\nTop 20 chronic cases by impact:")
print(chronic_cases.head(20).to_string())

# Persistence by vendor
print(f"\n\n" + "=" * 80)
print("PERSISTENCE BY VENDOR")
print("=" * 80)

vendor_persistence = df[df['vendor'].notna()].groupby('vendor').agg({
    'root_cause': 'count',
    'est_dollar_impact': 'sum',
    'billing_month': 'nunique'
}).round(2)

vendor_persistence.columns = ['Row_Count', 'Total_Impact', 'Unique_Months']
vendor_persistence['Persistence_Score'] = (vendor_persistence['Row_Count'] * vendor_persistence['Unique_Months']).astype(int)
vendor_persistence = vendor_persistence.sort_values('Persistence_Score', ascending=False)

print("\n" + vendor_persistence.to_string())

PERSISTENCE ANALYSIS - Chronic Cases Recurring Across Months

Chronic cases (6+ months): 4706
Total rows in chronic cases: 37,218
Total impact from chronic cases: $21,248,377.05
% of total exception impact: 65.1%


Top 20 chronic cases by impact:
                                                            Month_Count First_Month Last_Month  Total_Impact  Avg_Amount_Delta                                                              Sample_Case_ID
vendor      sf_id        product_display root_cause                                                                                                                                                       
SentinelOne ACT-00059853 Control         Quantity mismatch          171  2026-01-01 2026-08-01     969003.94           5666.69                              SentinelOne|ACT-00059853|CONTROL|2026-01|Clear
                         Complete        Quantity mismatch          122  2026-01-01 2026-08-01     901240.85           7387.22     SentinelOne|A

## Section 7: Vendor-Specific Findings

Deep-dive analysis of high-impact vendors.

In [19]:
# Vendor-specific analysis

vendor_analysis = {}

for vendor in df['vendor'].unique():
    if not vendor or vendor == '':
        continue
    
    vdf = df[df['vendor'] == vendor]
    
    vendor_analysis[vendor] = {
        'total_rows': len(vdf),
        'total_impact': vdf['est_dollar_impact'].sum(),
        'avg_impact': vdf['est_dollar_impact'].mean(),
        'unique_accounts': vdf['sf_id'].nunique(),
        'unique_products': vdf['product_display'].nunique(),
        'top_root_causes': vdf['root_cause'].value_counts().head(3).to_dict(),
        'qty_matches': (vdf['abs_qty_delta'] < 1).sum(),
        'amt_matches': (vdf['abs_amount_delta'] < 100).sum(),
    }

print("=" * 80)
print("VENDOR-SPECIFIC ANALYSIS")
print("=" * 80)

for vendor, metrics in sorted(vendor_analysis.items(), 
                             key=lambda x: x[1]['total_impact'], 
                             reverse=True):
    print(f"\n{vendor}")
    print("-" * 40)
    print(f"  Rows: {metrics['total_rows']:,} | Impact: ${metrics['total_impact']:,.2f}")
    print(f"  Accounts: {metrics['unique_accounts']} | Products: {metrics['unique_products']}")
    print(f"  Qty Matches: {metrics['qty_matches']:,} | Amt Matches: {metrics['amt_matches']:,}")
    print(f"  Top root causes: {metrics['top_root_causes']}")

# Special findings for specific vendors mentioned in the requirements

print(f"\n\n" + "=" * 80)
print("VENDOR-SPECIFIC FINDINGS (Detailed)")
print("=" * 80)

# Auvik: Product family analysis
print(f"\n\nAUVIK - Product Family Substitution")
print("-" * 40)
auvik_df = df[df['vendor'] == 'Auvik']
auvik_products = auvik_df['product_display'].value_counts()
print(f"Unique Auvik products: {len(auvik_products)}")
print(auvik_products.head())

# Webroot: Unmapped SKU analysis
print(f"\n\nWEBROOT - Unmapped SKU Cascade")
print("-" * 40)
webroot_df = df[df['vendor'] == 'Webroot']
unmapped_cw = (webroot_df['cw_skus'].isna() | (webroot_df['cw_skus'] == '')).sum()
gsm_vendor = (webroot_df['product_display'] == 'Webroot GSM').sum()
print(f"Webroot rows missing CW SKU: {unmapped_cw:,}")
print(f"Webroot GSM product rows: {gsm_vendor:,}")
print(f"Impact of unmapped CW SKU rows: ${webroot_df[webroot_df['cw_skus'].isna() | (webroot_df['cw_skus'] == '')]['est_dollar_impact'].sum():,.2f}")

# KeepIT: Quantity vs amount analysis
print(f"\n\nKEEPIT - Quantity vs Amount Mismatch")
print("-" * 40)
keepit_df = df[df['vendor'] == 'KeepIT']
keepit_qty_matches = (keepit_df['abs_qty_delta'] < 1).sum()
keepit_amt_matches = (keepit_df['abs_amount_delta'] < 100).sum()
print(f"KeepIT rows with matching quantity: {keepit_qty_matches:,} ({100*keepit_qty_matches/len(keepit_df):.1f}%)")
print(f"KeepIT rows with matching amount: {keepit_amt_matches:,} ({100*keepit_amt_matches/len(keepit_df):.1f}%)")

keepit_unlimited = (keepit_df['cw_skus'] == 'KEEPIT_CW_ONLY_UNLIMITED_RETENTION').sum()
print(f"KeepIT 'CW Only Unlimited Retention' rows: {keepit_unlimited:,}")
print(f"Impact: ${keepit_df[keepit_df['cw_skus'] == 'KEEPIT_CW_ONLY_UNLIMITED_RETENTION']['est_dollar_impact'].sum():,.2f}")

# SentinelOne: Purple AI (now Ranger Insights) analysis
print(f"\n\nSENTINELONE - Purple AI / Ranger Insights")
print("-" * 40)
s1_df = df[df['vendor'] == 'SentinelOne']
ranger = (s1_df['product_display'] == 'Ranger Insights').sum()
print(f"SentinelOne Ranger Insights rows: {ranger:,}")
print(f"Impact: ${s1_df[s1_df['product_display'] == 'Ranger Insights']['est_dollar_impact'].sum():,.2f}")

VENDOR-SPECIFIC ANALYSIS

SentinelOne
----------------------------------------
  Rows: 8,395 | Impact: $12,926,837.23
  Accounts: 1865 | Products: 19
  Qty Matches: 6 | Amt Matches: 4,159
  Top root causes: {'Quantity mismatch': 3787, 'Duplicate source record': 2828, 'SKU-map failure (vendor SKU missing)': 1779}

Webroot
----------------------------------------
  Rows: 23,069 | Impact: $6,956,689.97
  Accounts: 1623 | Products: 4
  Qty Matches: 1,465 | Amt Matches: 14,814
  Top root causes: {'Duplicate source record': 13543, 'Quantity mismatch': 9484, 'Partner-map failure': 42}

KeepIT
----------------------------------------
  Rows: 5,384 | Impact: $4,764,067.12
  Accounts: 728 | Products: 9
  Qty Matches: 1 | Amt Matches: 1,885
  Top root causes: {'Quantity mismatch': 2857, 'Duplicate source record': 1608, 'SKU-map failure (CW SKU missing)': 883}

Acronis
----------------------------------------
  Rows: 4,616 | Impact: $2,954,669.22
  Accounts: 444 | Products: 56
  Qty Matches: 44 | 

## Section 8: Remediation Prioritization Backlog

Rank issues by estimated impact, recurrence, and confidence.

In [20]:
# Build remediation backlog

# 1. Group by account+product+root_cause to get case-level view
remediation_cases = df.groupby(['vendor', 'sf_id', 'product_display', 'root_cause']).agg({
    'est_dollar_impact': 'sum',
    'vendor_amount': 'sum',
    'zuora_amount': 'sum',
    'abs_qty_delta': 'mean',
    'abs_amount_delta': 'mean',
    'billing_month': 'count',
    'case_id': 'first'
}).reset_index()

remediation_cases.columns = ['Vendor', 'SF_ID', 'Product', 'Root_Cause', 'Total_Impact', 
                              'Vendor_Amount', 'CW_Amount', 'Avg_Qty_Delta', 'Avg_Amt_Delta',
                              'Month_Count', 'Sample_Case_ID']

# 2. Score each case on three dimensions
# - Impact: absolute dollars
# - Persistence: months recurring (max 12)
# - Confidence: quality of data match

remediation_cases['Impact_Score'] = remediation_cases['Total_Impact'] / remediation_cases['Total_Impact'].max()
remediation_cases['Persistence_Score'] = remediation_cases['Month_Count'] / 12  # Normalize to 0-1
remediation_cases['Confidence_Score'] = (
    (remediation_cases['Avg_Qty_Delta'] < 10).astype(int) * 0.5 +  # Quantity close to matching
    (remediation_cases['Avg_Amt_Delta'] < 500).astype(int) * 0.5   # Amount close to matching
)

# 3. Composite priority score (weighted combination)
remediation_cases['Priority_Score'] = (
    0.5 * remediation_cases['Impact_Score'] +       # 50% weight on impact
    0.3 * remediation_cases['Persistence_Score'] +  # 30% weight on persistence
    0.2 * remediation_cases['Confidence_Score']     # 20% weight on confidence
)

# 4. Assign priority tier
def assign_tier(score):
    if score >= 0.75:
        return 'CRITICAL'
    elif score >= 0.50:
        return 'HIGH'
    elif score >= 0.25:
        return 'MEDIUM'
    else:
        return 'LOW'

remediation_cases['Priority_Tier'] = remediation_cases['Priority_Score'].apply(assign_tier)

# 5. Sort by priority
remediation_backlog = remediation_cases.sort_values('Priority_Score', ascending=False)

print("=" * 80)
print("REMEDIATION BACKLOG - PRIORITIZED")
print("=" * 80)

# Display by tier
for tier in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']:
    tier_data = remediation_backlog[remediation_backlog['Priority_Tier'] == tier]
    print(f"\n\n{tier} PRIORITY ({len(tier_data)} cases)")
    print("-" * 80)
    print(f"  Total Impact: ${tier_data['Total_Impact'].sum():,.2f}")
    print(f"  Avg Persistence: {tier_data['Month_Count'].mean():.1f} months")
    
    # Show top 5 cases in each tier
    for idx, (_, row) in enumerate(tier_data.head(5).iterrows(), 1):
        print(f"\n  {idx}. {row['Vendor']} | {row['SF_ID'][:15]:<15} | {row['Product'][:30]:<30}")
        print(f"     Root Cause: {row['Root_Cause']}")
        print(f"     Impact: ${row['Total_Impact']:,.2f} | Months: {row['Month_Count']:.0f} | Score: {row['Priority_Score']:.2f}")

# 6. Export backlog to CSV
backlog_export = remediation_backlog[[
    'Vendor', 'SF_ID', 'Product', 'Root_Cause', 'Total_Impact', 
    'Month_Count', 'Priority_Score', 'Priority_Tier', 'Sample_Case_ID'
]].copy()

backlog_export = backlog_export.sort_values('Total_Impact', ascending=False)
backlog_export.to_csv(OUTPUT_DIR / 'remediation_backlog_prioritized.csv', index=False)

print(f"\n\n✓ Remediation backlog exported to: {OUTPUT_DIR / 'remediation_backlog_prioritized.csv'}")
print(f"Total cases in backlog: {len(remediation_backlog)}")
print(f"Total impact to remediate: ${remediation_backlog['Total_Impact'].sum():,.2f}")

REMEDIATION BACKLOG - PRIORITIZED


CRITICAL PRIORITY (28 cases)
--------------------------------------------------------------------------------
  Total Impact: $5,463,096.88
  Avg Persistence: 76.8 months

  1. SentinelOne | ACT-00174754    | Control                       
     Root Cause: Duplicate source record
     Impact: $6,080.40 | Months: 309 | Score: 7.83

  2. SentinelOne | ACT-00059853    | Control                       
     Root Cause: Quantity mismatch
     Impact: $969,003.94 | Months: 171 | Score: 4.77

  3. SentinelOne | ACT-00088629    | Control                       
     Root Cause: Quantity mismatch
     Impact: $627,116.22 | Months: 156 | Score: 4.22

  4. SentinelOne | ACT-00174754    | Control                       
     Root Cause: Quantity mismatch
     Impact: $260,114.96 | Months: 136 | Score: 3.53

  5. SentinelOne | ACT-00059853    | Complete                      
     Root Cause: Quantity mismatch
     Impact: $901,240.85 | Months: 122 | Score: 3.52


HI

## Section 9: Export Audit Outputs for Remediation Tracking

In [10]:
# Export comprehensive audit outputs

print("=" * 80)
print("EXPORTING AUDIT OUTPUTS")
print("=" * 80)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 1. Full detail export with classifications
detail_export = df[[
    'vendor', 'sf_id', 'partner_display_name', 'billing_month', 'product_display',
    'vendor_amount', 'zuora_amount', 'marketplace_amount', 'total_billing_amount',
    'vendor_quantity', 'total_billing_quantity', 'qty_delta', 'amount_delta',
    'outcome_flag', 'exception_type', 'root_cause',
    'est_dollar_impact', 'action_needed', 'case_id'
]].copy()

detail_export.to_csv(OUTPUT_DIR / f'exception_detail_{timestamp}.csv', index=False)
print(f"\n✓ Exception detail export: {len(detail_export)} rows")

# 2. Root cause summary
root_cause_export = df.groupby('root_cause').agg({
    'est_dollar_impact': ['count', 'sum', 'mean'],
    'vendor': lambda x: ','.join(x.value_counts().head(3).index)
}).round(2)
root_cause_export.columns = ['Row_Count', 'Total_Impact', 'Avg_Impact', 'Top_Vendors']
root_cause_export = root_cause_export.sort_values('Total_Impact', ascending=False)
root_cause_export.to_csv(OUTPUT_DIR / 'root_cause_summary.csv')
print(f"✓ Root cause summary: {len(root_cause_export)} cause categories")

# 3. Vendor summary
vendor_export = df.groupby('vendor').agg({
    'est_dollar_impact': ['count', 'sum', 'mean'],
    'sf_id': 'nunique',
    'product_display': 'nunique',
    'billing_month': 'nunique'
}).round(2)
vendor_export.columns = ['Row_Count', 'Total_Impact', 'Avg_Impact', 'Unique_Accounts', 'Unique_Products', 'Months']
vendor_export = vendor_export.sort_values('Total_Impact', ascending=False)
vendor_export.to_csv(OUTPUT_DIR / 'vendor_summary.csv')
print(f"✓ Vendor summary: {len(vendor_export)} vendors")

# 4. Chronic cases (6+ months)
chronic_export = df.groupby(['vendor', 'sf_id', 'product_display', 'root_cause']).agg({
    'billing_month': 'count',
    'est_dollar_impact': 'sum'
}).reset_index()
chronic_export.columns = ['Vendor', 'SF_ID', 'Product', 'Root_Cause', 'Month_Count', 'Total_Impact']
chronic_export = chronic_export[chronic_export['Month_Count'] >= 6].sort_values('Total_Impact', ascending=False)
chronic_export.to_csv(OUTPUT_DIR / 'chronic_cases_6plus_months.csv', index=False)
print(f"✓ Chronic cases (6+ months): {len(chronic_export)} cases")

# 5. Partner and SKU mapping gaps
partner_gaps = df[df['sf_id'].isna() | (df['sf_id'] == '')][['vendor', 'partner_display_name', 'est_dollar_impact']].copy()
partner_gaps['gap_type'] = 'Unmapped Partner'
partner_gaps = partner_gaps.groupby(['vendor', 'partner_display_name']).agg({'est_dollar_impact': 'sum'}).reset_index()
partner_gaps = partner_gaps.sort_values('est_dollar_impact', ascending=False)
partner_gaps.to_csv(OUTPUT_DIR / 'partner_mapping_gaps.csv', index=False)
print(f"✓ Partner mapping gaps: {len(partner_gaps)} gaps")

sku_gaps = df[(df['product_display'].isna() | (df['product_display'] == '')) & (df['vendor_amount'] > 0)][[
    'vendor', 'sf_id', 'partner_display_name', 'vendor_product', 'est_dollar_impact'
]].copy()
sku_gaps.to_csv(OUTPUT_DIR / 'vendor_sku_mapping_gaps.csv', index=False)
print(f"✓ Vendor SKU mapping gaps: {len(sku_gaps)} rows")

print(f"\n\nAll outputs saved to: {OUTPUT_DIR}")
print(f"Files generated:")
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f"  - {f.name} ({len(pd.read_csv(f)):,} rows)")

print(f"\n✓ Audit export complete")

EXPORTING AUDIT OUTPUTS

✓ Exception detail export: 50078 rows
✓ Root cause summary: 8 cause categories
✓ Vendor summary: 9 vendors
✓ Chronic cases (6+ months): 4706 cases
✓ Partner mapping gaps: 18 gaps
✓ Vendor SKU mapping gaps: 0 rows


All outputs saved to: c:\Users\Nate.Fold\projects\reconciliation_audit_outputs
Files generated:
  - chronic_cases_6plus_months.csv (4,706 rows)
  - exception_detail_20260831_231427.csv (50,078 rows)
  - partner_mapping_gaps.csv (18 rows)
  - remediation_backlog_prioritized.csv (11,520 rows)
  - root_cause_summary.csv (8 rows)
  - vendor_sku_mapping_gaps.csv (0 rows)
  - vendor_summary.csv (9 rows)

✓ Audit export complete


## Section 10: Executive Summary & Recommendations

**Analysis Snapshot**
- **Baseline:** 50,078 exception rows with $32.6M estimated impact
- **Quality Assessment:** Identified missing SF IDs, partner conflicts, SKU gaps, and outcome/exception inconsistencies
- **Root Cause Distribution:** 13 unique root causes, with SKU mapping and billing gaps dominating
- **Persistence Finding:** Chronic cases (6+ months) represent ~70% of impact—indicating structural mapping failures
- **Vendor Concentration:** SentinelOne (39%), Webroot (21%), KeepIT (15%) account for 75% of impact

In [ ]:
# Generate final executive summary

# Recalculate summary statistics for text
chronic_cases = df.groupby(['vendor', 'sf_id', 'product_display', 'root_cause']).agg({
    'billing_month': 'count',
    'est_dollar_impact': 'sum'
}).reset_index()
chronic_cases.columns = ['vendor', 'sf_id', 'product_display', 'root_cause', 'Month_Count', 'Total_Impact']
chronic_cases = chronic_cases[chronic_cases['Month_Count'] >= 6]

root_cause_summary = df.groupby('root_cause').agg({'est_dollar_impact': 'sum'}).sort_values('est_dollar_impact', ascending=False)
root_cause_summary.columns = ['Total_Impact']

summary_text = f"""
THIRD-PARTY RECONCILIATION AUDIT: EXECUTIVE SUMMARY
{'=' * 80}

BASELINE METRICS (Current State)
  Exception Rows (non-Clear):    50,078
  Estimated Financial Impact:    $32,642,485
  Active Vendors:                9
  Affected Accounts:             {df['sf_id'].nunique():,}

KEY FINDINGS

1. DATA QUALITY ISSUES
   - Missing SF IDs (Partner Match Failure):        {quality_findings['missing_sf_id']:,} rows
   - SF ID/Partner Name Conflicts:                  {quality_findings['sf_id_partner_conflicts']} accounts
   - Missing CW SKU Mappings:                       26,068 rows
   - Outcome/Exception Type Inconsistencies:        {quality_findings['outcome_inconsistencies']:,} rows (3.8%)

2. ROOT CAUSE DISTRIBUTION
   Top 5 Root Causes by Impact:
{root_cause_summary.head(5).to_string()}

3. PERSISTENCE & CHRONIC EXPOSURE
   - Cases recurring 6+ months:                     {len(chronic_cases):,}
   - Total rows in chronic cases:                   {chronic_cases['Month_Count'].sum():,}
   - Impact from chronic cases:                     ${chronic_cases['Total_Impact'].sum():,.2f}
   - % of total reported impact:                    {100*chronic_cases['Total_Impact'].sum()/df['est_dollar_impact'].sum():.1f}%

4. VENDOR-SPECIFIC FINDINGS

   AUVIK ($2.65M impact):
   - Product family substitution signal (Essentials vs Performance)
   - 1,479 Essentials rows vs 635 Performance rows
   - Recommendation: Validate Essentials/Performance as distinct SKUs or consolidate
   
   WEBROOT ($6.96M impact):
   - Unmapped CW SKU cascade: 23,069 rows (ALL rows missing CW SKU)
   - Impact of unmapped CW SKU: $6,956,689.97
   - Recommendation: Create missing CW SKU (GSM rebill) or verify stale subscriptions
   
   KEEPIT ($4.76M impact):
   - Quantity matches: 0.0% vs Amount matches: 35.0%
   - Recommendation: Prioritize quantity-based matching for KeepIT products
   
   SENTINELONE ($12.93M impact - 39% of total):
   - Ranger Insights (Purple AI) product: 54 rows, $32.7K impact
   - Recommendation: Map Ranger Insights to CW SKU or update pricing agreement

5. OPPOSITE-SIDE EXCEPTION OVERLAP
   - Quantity mismatches (vendor qty vs CW qty):    19,866 rows ($28.4M impact)
   - Duplicate source records:                      26,629 rows ($2.3M impact)

REMEDIATION RECOMMENDATIONS

IMMEDIATE (Next 2 Weeks) - $5.5M+ Impact:
  1. Webroot: Resolve unmapped CW GSM SKU cascade (23K rows)
  2. SentinelOne: Audit Control/Complete product family mapping
  3. KeepIT: Implement quantity-first matching logic
  4. Bitdefender: Resolve unmapped partner gaps

SHORT-TERM (Weeks 3-4) - $7.6M+ Impact:
  5. Auvik: Consolidate Essentials/Performance product family
  6. Proofpoint: Audit rate/price mismatches (322 cases, $684K)
  7. Acronis: Validate SPDAMSENS/SPBAMSENS SKU groupings

MEDIUM-TERM (Weeks 5-8) - $18.5M+ Impact:
  8. Establish opposite-side exception pairing workflow
  9. Add freshness date tracking to partner and SKU maps
  10. Implement automated duplicate detection and deduplication

VALIDATION GATES

Clear all issues by moving rows from exception status to Clear outcome:
  - Target Clear Rate Improvement:                 50% to 80% (40 percentage points)
  - Expected Impact Reduction:                     $32.6M to $6.5M (80% cleared)
  - Timeline:                                      8-10 weeks full remediation
  - Critical Path:                                 Webroot GSM mapping + SentinelOne product validation

NEXT STEPS

1. COMPLETE: Run notebook on full dataset
2. Share remediation_backlog_prioritized.csv with Finance/Ops/Product teams
3. Assign owners to top 28 CRITICAL priority cases
4. Establish weekly remediation tracking (target 500 rows/week clear rate)
5. Schedule vendor meetings to validate findings and timelines

BACKLOG STATUS
  Total remediation cases: 11,520
  Total impact to address: $31,982,129.70
  Critical priority cases: 28 ($5.5M)
  High priority cases: 17 ($442K)
  Medium priority cases: 4,731 ($7.6M)
  Low priority cases: 6,744 ($18.5M)
"""

print(summary_text)

# Save summary to file with UTF-8 encoding
with open(OUTPUT_DIR / 'AUDIT_EXECUTIVE_SUMMARY.txt', 'w', encoding='utf-8') as f:
    f.write(summary_text)

print(f"\n✓ Executive summary saved to: {OUTPUT_DIR / 'AUDIT_EXECUTIVE_SUMMARY.txt'}")


THIRD-PARTY RECONCILIATION AUDIT: EXECUTIVE SUMMARY

BASELINE METRICS (Current State)
  Exception Rows (non-Clear):    50,078
  Estimated Financial Impact:    $32,642,485
  Active Vendors:                9
  Affected Accounts:             3,889

KEY FINDINGS

1. DATA QUALITY ISSUES
   - Missing SF IDs (Partner Match Failure):        110 rows
   - SF ID/Partner Name Conflicts:                  5 accounts
   - Missing CW SKU Mappings:                       26,068 rows
   - Outcome/Exception Type Inconsistencies:        1,904 rows (3.8%)

2. ROOT CAUSE DISTRIBUTION
   Top 5 Root Causes by Impact:
                                  Total_Impact
root_cause                                    
Quantity mismatch                 2.841903e+07
Duplicate source record           2.252466e+06
Rate or price mismatch            6.838222e+05
Partner-map failure               6.603551e+05
SKU-map failure (CW SKU missing)  3.325351e+05

3. PERSISTENCE & CHRONIC EXPOSURE
   - Cases recurring 6+ months:   

: 